# Simple inference

To run this notebook,, you need to first softlink the `tomato` dir to the path where the notebook is located.



## load the datasets

In [10]:
from argparse import Namespace
from tomato.data import GeneralFAD
from torch.utils.data import DataLoader

args = Namespace(
    data_path='example_data',
    test_batch_size=4,
    test_num_workers=1
)

dataset = GeneralFAD(args=args, split='test', train_mode=False)
dataloader = DataLoader(dataset, 
    batch_size=args.test_batch_size, 
    num_workers=args.test_num_workers,
    collate_fn=dataset.collate_fn)

first_batch = next(iter(dataloader))
print(first_batch)


fad_trim.py:INFO:36:Max length: 64000. Note: if the transformation is used, and the features are read from the disk, the max_len will be ignored, because the audios are already trimmed before saved
fad_trim.py:INFO:315:Loading dataset example_data, split: test
fad_trim.py:INFO:308:Dataset /mount/arbeitsdaten54/projekte/deepfake/fad/tomatoDD/example/example_data, split: test


{'uttids': ['MLAAD_en_jane_eyre_17_f000375', 'MLAAD_en_northandsouth_51_f000004', 'MLAAD_en_wives_and_daughters_27_f000024', 'MLAAD_en_pink_fairy_book_35_f000005'], 'feats': tensor([[[-0.0196,  0.0085,  0.0663,  ..., -0.0483, -0.0727, -0.0923]],

        [[-0.0828, -0.1024, -0.1216,  ...,  0.0488,  0.0543, -0.0868]],

        [[ 0.1390,  0.1318,  0.1258,  ..., -0.0049,  0.0075, -0.0137]],

        [[-0.1754, -0.1732, -0.1739,  ...,  0.0864,  0.1669,  0.2038]]]), 'labels': tensor([1, 1, 1, 1]), 'origin_ds': ['MLAAD_en', 'MLAAD_en', 'MLAAD_en', 'MLAAD_en'], 'speakers': ['unk', 'unk', 'unk', 'unk'], 'attackers': ['tts_models_en_ljspeech_fast_pitch', 'suno_bark', 'suno_bark', 'suno_bark'], 'padding_mask': None}


## load the model

In [25]:
from tomato.models import XLSRAdapter

mdl_args = Namespace(
    frontend='XLSR',
    frontend_path='/mount/arbeitsdaten54/projekte/deepfake/models/xlsr2_300m.pt',
    num_classes=1,
    model_class='XLSRAdapter',
    cuda=0
)
model = XLSRAdapter(mdl_args)
model.load_checkpoint('models/best.mdl')
model.eval()
print(model)



w2v2.py:INFO:31:Loading XLSR from /mount/arbeitsdaten54/projekte/deepfake/models/xlsr2_300m.pt
base.py:INFO:55:Freezing frontend model
frontend_only.py:INFO:30:Feature order: tnd
base.py:INFO:88:Loaded checkpoint from models/best.mdl


XLSRAdapter(
  (frontend_model): XLSR(
    (model): Wav2Vec2Model(
      (feature_extractor): ConvFeatureExtractionModel(
        (conv_layers): ModuleList(
          (0): Sequential(
            (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (1-4): 4 x Sequential(
            (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (1): Dropout(p=0.0, inplace=False)
            (2): Sequential(
              (0): TransposeLast()
              (1): Fp32LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (2): TransposeLast()
            )
            (3): GELU(approximate='none')
          )
          (5-6): 2 x Sequential(
            (0): Conv1d(512, 512, kerne

In [26]:
from tomato.criteria import OCSoftmax

loss_cfg = Namespace(
    model_class='OCSoftmax',
    fake_weight=1.0,
    real_weight=9.0
)
loss_fn = OCSoftmax(loss_cfg)
loss_fn.load_checkpoint('models/best.criterion.mdl')
print(loss_fn)


continual_learning_loss.py:INFO:69:weight for real and fake: 9.0, 1.0
continual_learning_loss.py:INFO:80:Loaded checkpoint from models/best.criterion.mdl


OCSoftmax(
  (softplus): Softplus(beta=1, threshold=20)
)


# infer

In [27]:
all_scores = []
all_labels = []
for batch in dataloader:
    output_dict = model(batch)
    loss, output_score, _ = loss_fn(batch, output_dict)
    all_scores.append(output_score.data.cpu())
    all_labels.append(batch['labels'].data.cpu())

print(all_scores)

[tensor([0.3108, 0.1783, 0.8139, 0.3404]), tensor([0.0526, 0.1518, 0.4720, 0.3559]), tensor([0.4587, 0.9714, 0.9918, 0.9713]), tensor([0.9841, 0.9824, 0.9937, 0.9873]), tensor([0.9673, 0.9860, 0.9904, 0.9697])]


In [29]:
import torch
label_tensor = torch.cat(all_labels, dim=0).numpy()
score_tensor = torch.cat(all_scores, dim=0).numpy()

from tomato.train_util import compute_eer
# score is cosine similarity score to the real center, so we need to reverse it
eer, thresh = compute_eer(label_tensor, -score_tensor)
print(f"EER: {eer*100:.2f}%, threshold: {thresh}")


EER: 10.00%, threshold: -0.9696957468986511
